Utilizzando lo script fornito durante la lezione, effettua una comparazione ingegneristica tra i due modelli.

Analisi Parametrica: stampa il sommario (model.summary()) di entrambi i modelli. Calcola la percentuale esatta di parametri in meno che possiede la versione GRU rispetto alla LSTM

Prova ad aggiungere ulteriori layer densi in uscita per migliorare il risultato, attenzione a bilanciare l'overfitting.

In [ ]:
import os

# 1. SETUP DEL BACKEND (Best Practice 2026)
# Keras 3 permette di scrivere codice una volta ed eseguirlo ovunque. 
# Forziamo PyTorch come motore matematico per sfruttare i suoi kernel ottimizzati.
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import torch
import pandas as pd
from keras import layers, ops
# Libreria standard 2026 per l'accesso immediato ai dati
from datasets import load_dataset 

# ==========================================
# 2. CARICAMENTO DATASET REALE
# ==========================================
print("[INFO] Caricamento recensioni da Amazon Polarity...")

try:
    # Identificativo completo: namespace/nome_dataset
    streaming_dataset = load_dataset(
        "fancyzhx/amazon_polarity",
        split="train",
        streaming=True
    )

    # Mescolamento approssimato tramite buffer.
    # Evita di prendere necessariamente solo le prime recensioni.
    streaming_dataset = streaming_dataset.shuffle(
        seed=42,
        buffer_size=10_000
    )

    # In streaming vengono scaricati soltanto i record necessari.
    rows = list(streaming_dataset.take(5000))

    if not rows:
        raise ValueError("Il dataset caricato è vuoto.")

    # Utilizziamo sia il titolo sia il contenuto della recensione.
    X_raw = [
        f"{row.get('title', '')} {row.get('content', '')}".strip()
        for row in rows
    ]

    y_raw = [row["label"] for row in rows]

    if len(X_raw) != len(y_raw):
        raise ValueError(
            f"Numero testi diverso dal numero etichette: "
            f"{len(X_raw)} testi, {len(y_raw)} etichette."
        )

    print(
        f"[INFO] Dataset pronto: "
        f"{len(X_raw)} campioni caricati con successo."
    )

except Exception as e:
    # Non proseguire con tre frasi inventate:
    # il training risultante sarebbe privo di valore.
    raise RuntimeError(
        f"Impossibile caricare Amazon Polarity: {e}"
    ) from e

# ==========================================
# 3. PREPROCESSING PIPELINE
# ==========================================
MAX_VOCAB_SIZE = 10000 
MAX_SEQUENCE_LENGTH = 150 

vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_SEQUENCE_LENGTH,
)

# [TEORIA] L'adattamento deve avvenire su un array di stringhe non vuoto.
# Assicuriamoci che ogni elemento sia una stringa (evita errori di tipo Float/None).
X_raw_clean = [str(text) for text in X_raw]

print("[INFO] Adattamento TextVectorization in corso...")
vectorizer.adapt(np.array(X_raw_clean))

# 1. Trasformazione in sequenze numeriche
X_seq = vectorizer(np.array(X_raw_clean))
y_seq = np.array(y_raw).astype("float32")

# 2. Risoluzione errore Device (GPU -> CPU)
# Portiamo i dati sulla CPU per poterli manipolare con NumPy
X_seq_numpy = X_seq.cpu().numpy() if hasattr(X_seq, "cpu") else X_seq.numpy()

# 3. DEFINIZIONE DEGLI INDICI (Risoluzione NameError)
# Teoria: Creiamo un array di numeri da 0 a N-1 e lo mescoliamo casualmente.
# Questo garantisce che X e y rimangano sincronizzati durante lo shuffling.
indices = np.arange(len(X_seq_numpy))
np.random.shuffle(indices)

# 4. Applicazione dello shuffling
X_seq_shuffled = X_seq_numpy[indices]
y_seq_shuffled = y_seq[indices]

# 5. Split Training/Test (80/20)
split_idx = int(len(X_seq_shuffled) * 0.8)
X_train, X_test = X_seq_shuffled[:split_idx], X_seq_shuffled[split_idx:]
y_train, y_test = y_seq_shuffled[:split_idx], y_seq_shuffled[split_idx:]

print(f"[INFO] Shuffle completato. Training set: {len(X_train)}, Test set: {len(X_test)}")

# ==========================================
# 4. DEFINIZIONE DEL MODELLO (LSTM vs GRU)
# ==========================================
# Teoria: Usiamo mask_zero=True. Questo dice alla RNN di ignorare i padding (0) 
# evitando di degradare il segnale del gradiente su frasi corte.
def build_sentiment_model(cell_type="lstm"):
    """
    Costruisce un modello RNN profondo con una testa di classificazione multi-layer.
    
    Teoria: L'aggiunta di layer densi aumenta la capacità del modello di combinare 
    le feature estratte dalla sequenza. La Batch Normalization assicura che 
    le attivazioni rimangano in un range ottimale, prevenendo la saturazione.
    """
    model = keras.Sequential([
        layers.Input(shape=(MAX_SEQUENCE_LENGTH,)),
        # Embedding: Mappa le parole in uno spazio vettoriale continuo.
        layers.Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, mask_zero=True),
        
        # RNN Bidirezionale: Estrae il contesto da entrambi i sensi della frase.
        layers.Bidirectional(layers.LSTM(64)) if cell_type == "lstm" else layers.Bidirectional(layers.GRU(64)),
        
        # --- INIZIO TESTA DEEP ---
        
        # Primo Layer Denso: Espande la capacità di rappresentazione.
        layers.Dense(128),
        layers.BatchNormalization(), # Teoria: Riduce l'Internal Covariate Shift velocizzando il training.
        layers.Activation("relu"),
        layers.Dropout(0.5), # Teoria: Spegne casualmente il 50% dei neuroni per evitare la co-adattazione.
        
        # Secondo Layer Denso: Inizia a comprimere le informazioni verso la decisione.
        layers.Dense(64),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Dropout(0.4), # Dropout leggermente inferiore man mano che ci avviciniamo all'output.
        
        # Terzo Layer Denso: Raffinamento finale delle feature semantiche.
        layers.Dense(32),
        layers.Activation("relu"),
        layers.Dropout(0.3),
        
        # --- FINE TESTA DEEP ---
        
        # Output: Sigmoid per classificazione binaria (Sentiment Positivo/Negativo).
        layers.Dense(1, activation='sigmoid')
    ])
    
    # AdamW è superiore nel gestire il Weight Decay rispetto ad Adam standard.
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-2),
        loss="binary_crossentropy", 
        metrics=["accuracy"]
    )
    return model

# ==========================================
# 5. BENCHMARKING E TRAINING
# ==========================================
print("\n--- Training LSTM Model (Amazon Real Data) ---")
lstm_model = build_sentiment_model("lstm")
# Usiamo EarlyStopping per evitare overfitting se la loss di validazione smette di scendere.
lstm_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=64)

print("\n--- Training GRU Model (Amazon Real Data) ---")
gru_model = build_sentiment_model("gru")
gru_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=64)

# ==========================================
# 6. SALVATAGGIO "BEST PRACTICE 2026"
# ==========================================
# Il formato .keras v3 include l'intera pipeline di preprocessing TextVectorization.
lstm_model.save("amazon_sentiment_pro_2026.keras")
print("\n[INFO] Modello salvato nel formato universale V3.")

# ==========================================
# 7. ANALISI DEI CASI DI ERRORE
# ==========================================
def analyze_errors(model, texts, labels):
    # Applichiamo la vettorizzazione in tempo reale per l'inferenza
    processed_input = vectorizer(np.array(texts))
    preds = model.predict(processed_input, verbose=0)
    preds_binary = (preds > 0.5).astype(int).flatten()
    
    print("\n--- ANALISI QUALITATIVA SUI DATI REALI ---")
    for i in range(len(texts)):
        status = "CORRETTO" if preds_binary[i] == labels[i] else "ERRORE"
        print(f"[{status}] Testo: {texts[i][:80]}...")
        print(f"Target: {labels[i]} | Pred: {preds_binary[i]} (Confidenza: {preds[i][0]:.2f})\n")

# Test su frasi nuove che contengono sfide semantiche (Negazioni e Sarcasmo)
real_world_tests = [
    "I expected much more from this software, a huge disappointment.", # Negativo
    "Surprisingly good, it actually solved all my bugs!",              # Positivo con parola 'bugs' (spesso negativa)
    "Great, another update that breaks everything. Thanks Amazon."    # Sarcasmo (Difficile)
]
test_labels = [0, 1, 0]

analyze_errors(lstm_model, real_world_tests, test_labels)